# 🏝 LvTS YouTube Downloader
**by LomaVata Tech Services — *Living With Technology*** · Raiwai, Suva, Fiji

📞 **833 1088** · **746 6941** &nbsp;|&nbsp; ✉️ lomavatatechfiji@gmail.com &nbsp;|&nbsp; 🌐 [fijianblood.github.io/lvts](https://fijianblood.github.io/lvts)
Facebook **@lvtsfiji** &nbsp;|&nbsp; TikTok **@fijianblood8**

---

### How to use (free, nothing to install)
1. Sign in with your Google account (top right).
2. Click **▶ Run** on **Step 1**, wait for **✅ Ready** (about 30 seconds). If Colab shows a warning about this notebook, click **Run anyway**.
3. In **Step 2**, paste a YouTube link, choose a quality, then click **▶ Run**.
4. Your browser downloads the file when it's done.

⚖️ **Please download only videos you own, have permission to download, or that are licensed for reuse (for example Creative Commons).** Downloading other videos may break YouTube's Terms of Service or copyright law.

*Need a computer fixed, a website, or a video made for your business? Contact LvTS.*


In [ ]:
#@title ① Step 1 – Set up the LvTS downloader (run once) { display-mode: "form" }
#@markdown Click **▶ Run** and wait for **✅ Ready**.
import subprocess, sys
from IPython.display import HTML, display

BRAND_CSS = """<style>
.lvts{font-family:system-ui,-apple-system,Segoe UI,Roboto,sans-serif;border-radius:16px;overflow:hidden;max-width:720px;
 box-shadow:0 6px 24px rgba(10,20,60,.25);margin:6px 0}
.lvts-top{background:linear-gradient(135deg,#0b1d4d 0%,#1e40af 60%,#4338ca 100%);color:#fff;padding:18px 22px}
.lvts-top h2{margin:0;font-size:22px;letter-spacing:.3px}
.lvts-top p{margin:4px 0 0;opacity:.9;font-size:14px}
.lvts-tag{display:inline-block;background:#f59e0b;color:#0b1d4d;font-weight:700;border-radius:999px;padding:2px 10px;font-size:12px;margin-bottom:8px}
.lvts-body{background:#fff;color:#0f172a;padding:14px 22px;font-size:14px}
.lvts-foot{background:#0b1d4d;color:#e2e8f0;padding:10px 22px;font-size:13px;line-height:1.6}
.lvts-foot a{color:#fbbf24;text-decoration:none}
.lvts-bar{height:12px;background:#e2e8f0;border-radius:999px;overflow:hidden;margin:8px 0}
.lvts-bar>div{height:100%;background:linear-gradient(90deg,#f59e0b,#fbbf24);transition:width .3s}
.ok{color:#047857;font-weight:700}.err{color:#b91c1c;font-weight:700}
</style>"""
FOOTER = """<div class="lvts-foot">📞 <b>833 1088</b> · <b>746 6941</b> &nbsp;|&nbsp; ✉️ <a href="mailto:lomavatatechfiji@gmail.com">lomavatatechfiji@gmail.com</a><br>
🌐 <a href="https://fijianblood.github.io/lvts" target="_blank">fijianblood.github.io/lvts</a> &nbsp;|&nbsp; Facebook <b>@lvtsfiji</b> &nbsp;|&nbsp; TikTok <b>@fijianblood8</b></div>"""

def lvts_card(body_html):
    return HTML(BRAND_CSS + '<div class="lvts"><div class="lvts-top"><span class="lvts-tag">JS 🏝 LvTS</span>'
                '<h2>LvTS YouTube Downloader</h2><p>LomaVata Tech Services · Living With Technology · Suva, Fiji</p></div>'
                '<div class="lvts-body">' + body_html + '</div>' + FOOTER + '</div>')

status = display(lvts_card("⏳ Installing the download tools… (about 30 seconds)"), display_id=True)
# yt-dlp[default] includes yt-dlp-ejs; deno is the JavaScript runtime yt-dlp needs for YouTube
result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "yt-dlp[default]", "deno"],
                        capture_output=True, text=True)
if result.returncode != 0:
    status.update(lvts_card('<span class="err">❌ Setup failed.</span><pre>' + result.stderr[-1500:] + '</pre>'))
else:
    import importlib, yt_dlp
    importlib.reload(yt_dlp)
    deno_ok = subprocess.run(["deno", "--version"], capture_output=True).returncode == 0
    ffmpeg_ok = subprocess.run(["ffmpeg", "-version"], capture_output=True).returncode == 0
    if deno_ok and ffmpeg_ok:
        status.update(lvts_card('<span class="ok">✅ Ready!</span> Now go to <b>Step 2</b>, paste a YouTube link and click ▶ Run.'
                                '<br><small>yt-dlp ' + yt_dlp.version.__version__ + '</small>'))
    else:
        status.update(lvts_card('<span class="err">❌ Missing part:</span> ' + ('' if deno_ok else 'Deno ') +
                                ('' if ffmpeg_ok else 'ffmpeg') + '. Try Runtime → Restart session, then run Step 1 again.'))


In [ ]:
#@title ② Step 2 – Download a video { display-mode: "form" }
video_url = "" #@param {type:"string"}
quality = "720p (recommended)" #@param ["1080p", "720p (recommended)", "480p", "Best available (largest file)", "Audio only (MP3)"]
save_copy_to_google_drive = False #@param {type:"boolean"}
#@markdown ⚖️ Only download videos you own, have permission for, or that are licensed for reuse.

import os, re, glob, html
from IPython.display import display

try:
    lvts_card
except NameError:
    raise SystemExit("Please run Step 1 first.")

import yt_dlp
from yt_dlp.utils import DownloadError

DOWNLOAD_DIR = "/content/LvTS_Downloads"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

class _QuietLogger:  # we show our own friendly messages instead of yt-dlp's raw text
    def debug(self, msg): pass
    def info(self, msg): pass
    def warning(self, msg): pass
    def error(self, msg): pass

def build_options(choice):
    opts = {
        "logger": _QuietLogger(),
        "outtmpl": os.path.join(DOWNLOAD_DIR, "%(title).90B.%(ext)s"),  # file named after the video title
        "noplaylist": True,          # only this video, even if the link is inside a playlist
        "quiet": True, "no_warnings": True, "noprogress": True,
    }
    heights = {"1080p": 1080, "720p (recommended)": 720, "480p": 480}
    if choice in heights:
        h = heights[choice]
        # Prefer H.264 + AAC so the MP4 plays on phones, TVs and Windows
        opts["format"] = (f"bestvideo[height<={h}][vcodec^=avc1]+bestaudio[ext=m4a]/"
                          f"bestvideo[height<={h}]+bestaudio/best[height<={h}]/best")
        opts["merge_output_format"] = "mp4"
    elif choice.startswith("Audio"):
        opts["format"] = "bestaudio/best"
        opts["postprocessors"] = [{"key": "FFmpegExtractAudio", "preferredcodec": "mp3", "preferredquality": "192"}]
    else:
        opts["format"] = "bestvideo+bestaudio/best"
        opts["merge_output_format"] = "mp4"
    return opts

def bar(pct, text):
    return lvts_card(f'{text}<div class="lvts-bar"><div style="width:{pct:.0f}%"></div></div>')

url = video_url.strip()
if not re.match(r"^https?://(www\.|m\.|music\.)?(youtube\.com|youtu\.be)/", url):
    display(lvts_card('<span class="err">Please paste a YouTube link</span> (it starts with https://www.youtube.com/ or https://youtu.be/) into <b>video_url</b>, then click ▶ Run again.'))
else:
    panel = display(bar(0, "⏳ Getting video details…"), display_id=True)
    state = {"title": ""}

    def on_progress(d):
        state["title"] = d.get("info_dict", {}).get("title", state["title"])
        if d["status"] == "downloading":
            total = d.get("total_bytes") or d.get("total_bytes_estimate") or 0
            done = d.get("downloaded_bytes", 0)
            pct = done * 100 / total if total else 0
            mb = f"{done/1e6:.1f} MB" + (f" of {total/1e6:.1f} MB" if total else "")
            panel.update(bar(pct, f"⬇️ Downloading <b>{html.escape(state['title'])}</b><br><small>{mb}</small>"))
        elif d["status"] == "finished":
            panel.update(bar(100, "🔧 Finishing the file (joining video and sound)…"))

    opts = build_options(quality)
    opts["progress_hooks"] = [on_progress]
    try:
        with yt_dlp.YoutubeDL(opts) as ydl:
            info = ydl.extract_info(url, download=True)
        paths = [d.get("filepath") for d in info.get("requested_downloads", []) if d.get("filepath")]
        path = paths[0] if paths else max(glob.glob(DOWNLOAD_DIR + "/*"), key=os.path.getmtime)
        size_mb = os.path.getsize(path) / 1e6
        msg = (f'<span class="ok">✅ Done!</span> <b>{html.escape(os.path.basename(path))}</b> ({size_mb:.1f} MB)<br>'
               'Your browser will now download the file. Big files can take a few minutes to arrive.')
        if save_copy_to_google_drive:
            from google.colab import drive
            import shutil
            drive.mount("/content/drive")
            dest = "/content/drive/MyDrive/LvTS_Downloads"
            os.makedirs(dest, exist_ok=True)
            shutil.copy(path, dest)
            msg += "<br>📁 A copy was also saved to <b>Google Drive → LvTS_Downloads</b>."
        panel.update(lvts_card(msg))
        from google.colab import files
        files.download(path)
    except DownloadError as e:
        text = str(e)
        if "confirm you" in text and "bot" in text:
            hint = ("YouTube is blocking this Colab computer right now. Try again later, or click "
                    "<b>Runtime → Disconnect and delete runtime</b>, then run Step 1 and Step 2 again to get a different computer.")
        elif "Private video" in text or "members" in text.lower():
            hint = "This video is private or members-only, so it can't be downloaded."
        else:
            hint = "Check the link and try again. If it keeps failing, run Step 1 again to update the tools."
        panel.update(lvts_card(f'<span class="err">❌ Download failed.</span> {hint}<br><small>{html.escape(text[:300])}</small>'))
